<a href="https://colab.research.google.com/github/catdlia/GCI-world/blob/main/final.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Імпорти та Базове налаштування

In [14]:
!pip install catboost optuna lightgbm xgboost


In [15]:
import os
import warnings
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Моделі
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
from sklearn.ensemble import RandomForestClassifier

# Валідація та Оптимізація
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# Шляхи (переконайтеся, що вони правильні для вашого Google Drive)
DATA_DIR = Path('/content/drive/MyDrive/GCI/competition')
train = pd.read_csv(DATA_DIR / 'input/train.csv')
test = pd.read_csv(DATA_DIR / 'input/test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'input/sample_submission.csv')

y = train['Drafted']
X_train_raw = train.drop(['Drafted'], axis=1)

Глобальний Препроцесинг та Доменні фічі (Domain Knowledge)

In [16]:
# Залишаємо сиру школу для CatBoost!
df['School_Raw'] = df['School']

num_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
df['Total_NaNs'] = df[num_cols].isnull().sum(axis=1)
for col in num_cols:
    df[f'is_missing_{col}'] = df[col].isnull().astype(int)

df['BMI'] = df['Weight'] / (df['Height'] ** 2)
df['Speed_Score'] = (df['Weight'] * 200) / (df['Sprint_40yd'] ** 4)
df['Explosion'] = df['Vertical_Jump'] + df['Broad_Jump']

# Рахуємо Z-Score за Position_Type (щоб уникнути шуму від рідкісних позицій)
for col in num_cols + ['Weight', 'Height', 'BMI', 'Speed_Score']:
    df[f'{col}_PosType_Z'] = df.groupby('Position_Type')[col].transform(lambda x: (x - x.mean()) / (x.std() + 1e-6))
    df[f'{col}_PosType_Z'] = df[f'{col}_PosType_Z'].fillna(0)

# Чистка перед навчанням
cols_to_drop = ['is_train', 'Id', 'Player_Type']
X_train_clean = df[df['is_train'] == 1].drop(cols_to_drop, axis=1)
X_test_clean = df[df['is_train'] == 0].drop(cols_to_drop, axis=1)

# Типизація
cat_features = ['Position_Type', 'Position', 'School_Raw']
for col in cat_features:
    X_train_clean[col] = X_train_clean[col].astype('category')
    X_test_clean[col] = X_test_clean[col].astype('category')

Налаштування моделей та Monotonic Constraints

In [17]:
import os
import warnings
from pathlib import Path
import pandas as pd
import numpy as np

# Моделі
from catboost import CatBoostClassifier, Pool
import xgboost as xgb
from lightgbm import LGBMClassifier

# Валідація
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.metrics import roc_auc_score
from scipy.optimize import minimize

warnings.filterwarnings('ignore')
RANDOM_STATE = 42

# Шляхи (переконайтеся, що правильні)
DATA_DIR = Path('/content/drive/MyDrive/GCI/competition')
train = pd.read_csv(DATA_DIR / 'input/train.csv')
test = pd.read_csv(DATA_DIR / 'input/test.csv')
sample_sub = pd.read_csv(DATA_DIR / 'input/sample_submission.csv')

y = train['Drafted']
X_train_raw = train.drop(['Drafted'], axis=1)

# Об'єднуємо для обробки
X_train_raw['is_train'] = 1
test['is_train'] = 0
df = pd.concat([X_train_raw, test], ignore_index=True)

# 1. Заповнення пропусків у категоріях
cat_features = ['School', 'Position_Type', 'Position', 'Player_Type']
for col in cat_features:
    df[col] = df[col].fillna('Unknown').astype(str)

# 2. Угруповання рідкісних шкіл (КРИТИЧНО ВАЖЛИВО для CatBoost)
school_counts = df['School'].value_counts()
rare_schools = school_counts[school_counts < 5].index
df.loc[df['School'].isin(rare_schools), 'School'] = 'Rare_School'
df['School_Raw'] = df['School'] # Зберігаємо для CatBoost

# 3. Магія пропусків
num_cols = ['Sprint_40yd', 'Vertical_Jump', 'Bench_Press_Reps', 'Broad_Jump', 'Agility_3cone', 'Shuttle']
df['Total_NaNs'] = df[num_cols].isnull().sum(axis=1)
for col in num_cols:
    df[f'is_missing_{col}'] = df[col].isnull().astype(int)

# 4. Ваші "Золоті" доменні фічі
df['BMI'] = df['Weight'] / ((df['Height'] + 1e-6) ** 2)
df['Speed_Score'] = (df['Weight'] * 200) / ((df['Sprint_40yd'] + 1e-6) ** 4)
df['Jump_Power'] = df['Vertical_Jump'] * df['Weight'] # Інтеграція маси і стрибка
df['Explosion'] = df['Vertical_Jump'] + df['Broad_Jump']

# 5. Pos_Diff (Різниця від середнього по позиції - замість Z-score)
for col in num_cols + ['Weight', 'Height', 'BMI']:
    # Різниця від середнього для конкретного Position_Type
    df[f'{col}_Pos_Diff'] = df.groupby('Position_Type')[col].transform(lambda x: x - x.mean())
    df[f'{col}_Pos_Diff'] = df[f'{col}_Pos_Diff'].fillna(0)

# Чистка перед навчанням
cols_to_drop = ['is_train', 'Id', 'Player_Type']
X_train_clean = df[df['is_train'] == 1].drop(cols_to_drop, axis=1)
X_test_clean = df[df['is_train'] == 0].drop(cols_to_drop, axis=1)

# Типизація
for col in ['Position_Type', 'Position', 'School_Raw']:
    X_train_clean[col] = X_train_clean[col].astype('category')
    X_test_clean[col] = X_test_clean[col].astype('category')

print(f"Готово! Розмірність трейну: {X_train_clean.shape}")

Готово! Розмірність трейну: (2781, 34)


Repeated K-Fold, Target Encoding та OOF Генерація

In [18]:
# Тільки 100% надійні монотонні обмеження
mono_constraints = {
    'Sprint_40yd': -1, 'Vertical_Jump': 1, 'Broad_Jump': 1, 'Bench_Press_Reps': 1,
    'Explosion': 1, 'Jump_Power': 1
}

# ВАШІ ПАРАМЕТРИ З OPTUNA
cb_params = {
    'iterations': 1787,
    'learning_rate': 0.06874094281549652,
    'depth': 3,
    'l2_leaf_reg': 5.290355672999665,
    'random_strength': 11.502343031921322,
    'bagging_temperature': 0.4122157832620809,
    'border_count': 154,
    'eval_metric': 'AUC',
    'random_seed': RANDOM_STATE,
    'verbose': False
}

xgb_params = {
    'n_estimators': 2372,
    'learning_rate': 0.010081751720435782,
    'max_depth': 8,
    'min_child_weight': 5,
    'subsample': 0.8434336534358337,
    'colsample_bytree': 0.6396772098220731,
    'gamma': 6.228982752620394,
    'reg_alpha': 0.0011212515235573147,
    'reg_lambda': 0.0008962757039321162,
    'tree_method': 'hist',
    'enable_categorical': True,
    'monotone_constraints': mono_constraints,
    'random_state': RANDOM_STATE,
    'n_jobs': -1
}

lgb_params = {
    'n_estimators': 1340,
    'learning_rate': 0.007498537342889904,
    'num_leaves': 206,
    'max_depth': 7,
    'min_child_samples': 93,
    'subsample': 0.6783036995536487,
    'colsample_bytree': 0.4001429737830276,
    'min_split_gain': 0.0021760215422934235,
    'reg_alpha': 0.13088969185671115,
    'reg_lambda': 0.008820233860660045,
    'random_state': RANDOM_STATE,
    'n_jobs': -1,
    'verbose': -1
}

N_SPLITS = 5
N_REPEATS = 3
rskf = RepeatedStratifiedKFold(n_splits=N_SPLITS, n_repeats=N_REPEATS, random_state=RANDOM_STATE)

oof_cb, test_cb = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))
oof_xgb, test_xgb = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))
oof_lgb, test_lgb = np.zeros(len(X_train_clean)), np.zeros(len(X_test_clean))

print("🚀 Починаємо навчання...")

for fold, (train_idx, valid_idx) in enumerate(rskf.split(X_train_clean, y)):
    X_tr, y_tr = X_train_clean.iloc[train_idx].copy(), y.iloc[train_idx]
    X_va, y_va = X_train_clean.iloc[valid_idx].copy(), y.iloc[valid_idx]
    X_te = X_test_clean.copy()

    # --- Target Encoding для School (Тільки на базі X_tr!) ---
    school_means = y_tr.groupby(X_tr['School']).mean()
    global_mean = y_tr.mean()
    for df_temp in [X_tr, X_va, X_te]:
        # Згладжування: якщо школи немає в трейні, даємо global_mean
        df_temp['School_TE'] = df_temp['School'].map(school_means).fillna(global_mean)
        df_temp.drop(['School'], axis=1, inplace=True)

    # --- 1. CatBoost (Бере School_Raw) ---
    cat_cols_cb = ['Position_Type', 'Position', 'School_Raw']
    X_tr_cb, X_va_cb, X_te_cb = X_tr.drop(['School_TE'], axis=1), X_va.drop(['School_TE'], axis=1), X_te.drop(['School_TE'], axis=1)

    model_cb = CatBoostClassifier(**cb_params)
    model_cb.fit(Pool(X_tr_cb, y_tr, cat_features=cat_cols_cb),
                 eval_set=Pool(X_va_cb, y_va, cat_features=cat_cols_cb),
                 early_stopping_rounds=150)
    oof_cb[valid_idx] += model_cb.predict_proba(X_va_cb)[:, 1] / N_REPEATS
    test_cb += model_cb.predict_proba(X_te_cb)[:, 1] / (N_SPLITS * N_REPEATS)

    # --- 2. XGBoost (Бере School_TE) ---
    X_tr_trees, X_va_trees, X_te_trees = X_tr.drop(['School_Raw'], axis=1), X_va.drop(['School_Raw'], axis=1), X_te.drop(['School_Raw'], axis=1)

    model_xgb = xgb.XGBClassifier(**xgb_params)
    model_xgb.fit(X_tr_trees, y_tr, eval_set=[(X_va_trees, y_va)], verbose=False)
    oof_xgb[valid_idx] += model_xgb.predict_proba(X_va_trees)[:, 1] / N_REPEATS
    test_xgb += model_xgb.predict_proba(X_te_trees)[:, 1] / (N_SPLITS * N_REPEATS)

    # --- 3. LightGBM (Бере School_TE) ---
    model_lgb = LGBMClassifier(**lgb_params)
    model_lgb.fit(X_tr_trees, y_tr, eval_set=[(X_va_trees, y_va)])
    oof_lgb[valid_idx] += model_lgb.predict_proba(X_va_trees)[:, 1] / N_REPEATS
    test_lgb += model_lgb.predict_proba(X_te_trees)[:, 1] / (N_SPLITS * N_REPEATS)

# --- Оптимізація ваг (Scipy Nelder-Mead) ---
def auc_loss_func(weights):
    weights = np.array(weights) / np.sum(weights)
    blend_oof = (weights[0] * oof_cb) + (weights[1] * oof_xgb) + (weights[2] * oof_lgb)
    return 1.0 - roc_auc_score(y, blend_oof)

res = minimize(auc_loss_func, [0.33, 0.33, 0.33], method='Nelder-Mead')
bw = res.x / np.sum(res.x)

print("\n" + "="*40)
print(f"🌲 CatBoost OOF AUC: {roc_auc_score(y, oof_cb):.5f}")
print(f"⚡ XGBoost OOF AUC: {roc_auc_score(y, oof_xgb):.5f}")
print(f"💡 LightGBM OOF AUC: {roc_auc_score(y, oof_lgb):.5f}")
print("="*40)
print(f"💎 Ідеальні ваги: CB={bw[0]:.1%}, XGB={bw[1]:.1%}, LGBM={bw[2]:.1%}")
final_oof = (bw[0] * oof_cb) + (bw[1] * oof_xgb) + (bw[2] * oof_lgb)
print(f"🏆 ФІНАЛЬНИЙ ENSEMBLE OOF AUC: {roc_auc_score(y, final_oof):.5f}")

# САБМІТ
final_test_preds = (bw[0] * test_cb) + (bw[1] * test_xgb) + (bw[2] * test_lgb)
submission = sample_sub.copy()
submission['Drafted'] = final_test_preds
OUTPUT_PATH = DATA_DIR / 'final_push_submission.csv'
submission.to_csv(OUTPUT_PATH, index=False)
print(f"✅ Збережено фінальний сабміт: {OUTPUT_PATH.name}")

🚀 Починаємо навчання...

🌲 CatBoost OOF AUC: 0.84183
⚡ XGBoost OOF AUC: 0.83395
💡 LightGBM OOF AUC: 0.82814
💎 Ідеальні ваги: CB=80.8%, XGB=33.3%, LGBM=-14.1%
🏆 ФІНАЛЬНИЙ ENSEMBLE OOF AUC: 0.84328
✅ Збережено фінальний сабміт: final_push_submission.csv
